In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error
import os
import pandas as pd

In [2]:
years = [2008, 2009, 2010, 2011, 2012]

output_dir = "/home/dtaneja/analysis-dishika/notebooks"
os.makedirs(output_dir, exist_ok=True)

In [3]:
hrdps_processed_files = []
for year in years:
    file = f"{output_dir}/HRDPS_{year}_tair_3h_with_latlon.nc"
    hrdps_processed_files.append(file)
ds_hrdps = xr.open_mfdataset(hrdps_processed_files,combine="by_coords")
ds_hrdps = ds_hrdps.sortby("time_counter")
print(ds_hrdps)

/tmp/ipykernel_1758729/259932316.py:5: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds_hrdps = xr.open_mfdataset(hrdps_processed_files,combine="by_coords")


<xarray.Dataset> Size: 12GB
Dimensions:       (time_counter: 14592, y: 266, x: 256)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 117kB 2008-01-01 ... 2012-12-...
    nav_lat       (time_counter, y, x) float32 4GB 45.61 45.62 ... 52.37 52.37
    nav_lon       (time_counter, y, x) float32 4GB 232.7 232.7 ... 239.6 239.7
Dimensions without coordinates: y, x
Data variables:
    tair          (time_counter, y, x) float32 4GB dask.array<chunksize=(2912, 266, 256), meta=np.ndarray>


In [4]:
print("First timestamp:", ds_hrdps.time_counter.values[0])
print("Last timestamp: ", ds_hrdps.time_counter.values[-1])

First timestamp: 2008-01-01T00:00:00.000000000
Last timestamp:  2012-12-31T21:00:00.000000000


In [5]:
# Training: 2009, 2010, and 2011
ds_hrdps_train = ds_hrdps.sel(time_counter=slice("2009-01-01", "2011-12-31"))
# Validation: 2008
ds_hrdps_val = ds_hrdps.sel(time_counter=slice("2008-01-01", "2008-12-31"))
# Testing: 2012
ds_hrdps_test = ds_hrdps.sel(time_counter=slice("2012-01-01", "2012-12-31"))
print("HRDPS Training:")
print(ds_hrdps_train.time_counter.values[0],"to",ds_hrdps_train.time_counter.values[-1])
print("HRDPS Validation:")
print(ds_hrdps_val.time_counter.values[0],"to",ds_hrdps_val.time_counter.values[-1])
print("HRDPS Testing:")
print(ds_hrdps_test.time_counter.values[0],"to",ds_hrdps_test.time_counter.values[-1])

HRDPS Training:
2009-01-01T00:00:00.000000000 to 2011-12-31T21:00:00.000000000
HRDPS Validation:
2008-01-01T00:00:00.000000000 to 2008-12-31T21:00:00.000000000
HRDPS Testing:
2012-01-01T00:00:00.000000000 to 2012-12-31T21:00:00.000000000


In [6]:
canrcm_files = sorted(glob.glob("/results/forcing/CanRCM5/*.nc"))

In [7]:
canrcm_years = [2008, 2009, 2010, 2011, 2012]
canrcm_tas_files = []
for year in canrcm_years:
    matches = sorted(glob.glob(f"/results/forcing/CanRCM5/"f"*_{year}01_{year}12_3h_tas.nc"))
    canrcm_tas_files.append(matches[0])

print("CanRCM temperature files:")
for file in canrcm_tas_files:
    print(file)

CanRCM temperature files:
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200801_200812_3h_tas.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200901_200912_3h_tas.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201001_201012_3h_tas.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201101_201112_3h_tas.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201201_201212_3h_tas.nc


In [8]:
from datetime import timedelta
ds_canrcm = xr.open_mfdataset(canrcm_tas_files,combine="by_coords")
ds_canrcm = ds_canrcm.sortby("time")
ds_canrcm = ds_canrcm.assign_coords(time=ds_canrcm.time.values - timedelta(hours=3))
print(ds_canrcm)

<xarray.Dataset> Size: 7GB
Dimensions:       (time: 14600, bnds: 2, rlat: 320, rlon: 360)
Coordinates:
  * time          (time) object 117kB 2008-01-01 00:00:00 ... 2012-12-31 21:0...
  * rlat          (rlat) float64 3kB -35.11 -34.89 -34.67 ... 34.63 34.85 35.07
  * rlon          (rlon) float64 3kB -39.27 -39.05 -38.83 ... 39.27 39.49 39.71
    lon           (rlat, rlon) float64 922kB dask.array<chunksize=(320, 360), meta=np.ndarray>
    lat           (rlat, rlon) float64 922kB dask.array<chunksize=(320, 360), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    time_bnds     (time, bnds) object 234kB dask.array<chunksize=(2920, 2), meta=np.ndarray>
    rotated_pole  (time) |S1 15kB b'' b'' b'' b'' b'' ... b'' b'' b'' b'' b''
    tas           (time, rlat, rlon) float32 7GB dask.array<chunksize=(2920, 320, 360), meta=np.ndarray>
Attributes: (12/17)
    title:                          CanRCM4 model output prepared for CORDEX ...
    institution:                    

/tmp/ipykernel_1758729/3930337657.py:2: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_canrcm = xr.open_mfdataset(canrcm_tas_files,combine="by_coords")


In [9]:
lat_hr = ds_hrdps["nav_lat"]
lon_hr = ds_hrdps["nav_lon"]
lon_hr_normalized = ((lon_hr + 180) % 360) - 180
lat_min = float(lat_hr.min())
lat_max = float(lat_hr.max())
lon_min = float(lon_hr_normalized.min())
lon_max = float(lon_hr_normalized.max())
print("HRDPS latitude range:")
print(lat_min, "to", lat_max)
print("HRDPS longitude range:")
print(lon_min, "to", lon_max)

HRDPS latitude range:
45.52116012573242 to 52.4603157043457
HRDPS longitude range:
-129.47222900390625 to -119.20327758789062


In [10]:
lat_lr = ds_canrcm["lat"]
lon_lr = ds_canrcm["lon"]
lon_lr_normalized = ((lon_lr + 180) % 360) - 180
mask_lr = ((lat_lr >= lat_min) &(lat_lr <= lat_max) &(lon_lr_normalized >= lon_min) &(lon_lr_normalized <= lon_max))
i_idx, j_idx = np.where(mask_lr.values)
i_min = i_idx.min()
i_max = i_idx.max()
j_min = j_idx.min()
j_max = j_idx.max()
print("CanRCM grid indices:")
print("rlat:", i_min, "to", i_max)
print("rlon:", j_min, "to", j_max)

CanRCM grid indices:
rlat: 161 to 200
rlon: 79 to 117


In [11]:
ds_canrcm_cut = (ds_canrcm[["tas"]].isel(rlat=slice(i_min, i_max + 1),rlon=slice(j_min, j_max + 1)))
print(ds_canrcm_cut)

<xarray.Dataset> Size: 91MB
Dimensions:  (time: 14600, rlat: 40, rlon: 39)
Coordinates:
  * time     (time) object 117kB 2008-01-01 00:00:00 ... 2012-12-31 21:00:00
  * rlat     (rlat) float64 320B 0.31 0.53 0.75 0.97 ... 8.23 8.45 8.67 8.89
  * rlon     (rlon) float64 312B -21.89 -21.67 -21.45 ... -13.97 -13.75 -13.53
    lon      (rlat, rlon) float64 12kB dask.array<chunksize=(40, 39), meta=np.ndarray>
    lat      (rlat, rlon) float64 12kB dask.array<chunksize=(40, 39), meta=np.ndarray>
Data variables:
    tas      (time, rlat, rlon) float32 91MB dask.array<chunksize=(2920, 40, 39), meta=np.ndarray>
Attributes: (12/17)
    title:                          CanRCM4 model output prepared for CORDEX ...
    institution:                    CCCma (Canadian Centre for Climate Modell...
    institute_id:                   CCCma
    driving_experiment:             , , r1i1p1
    driving_model_ensemble_member:  r1i1p1
    realization:                    1
    ...                             ..

In [12]:
# Training: 2009, 2010, and 2011
ds_canrcm_train = ds_canrcm_cut.sel(time=slice("2009-01-01", "2011-12-31"))
# Validation: 2008
ds_canrcm_val = ds_canrcm_cut.sel(time=slice("2008-01-01", "2008-12-31"))
# Testing: 2012
ds_canrcm_test = ds_canrcm_cut.sel(time=slice("2012-01-01", "2012-12-31"))
print("CanRCM Training:")
print(ds_canrcm_train.time.values[0], "to", ds_canrcm_train.time.values[-1])
print("CanRCM Validation:")
print(ds_canrcm_val.time.values[0], "to", ds_canrcm_val.time.values[-1])
print("CanRCM Testing:")
print(ds_canrcm_test.time.values[0], "to", ds_canrcm_test.time.values[-1])

CanRCM Training:
2009-01-01 00:00:00 to 2011-12-31 21:00:00
CanRCM Validation:
2008-01-01 00:00:00 to 2008-12-31 21:00:00
CanRCM Testing:
2012-01-01 00:00:00 to 2012-12-31 21:00:00


In [13]:
def time_to_string(value):
    if isinstance(value, np.datetime64):
        return np.datetime_as_string(value,unit="s").replace("T", " ")
    if hasattr(value, "strftime"):
        return value.strftime("%Y-%m-%d %H:%M:%S")
    raise TypeError(f"Unsupported time type: {type(value)}")

def align_time_pair(ds_canrcm_split, ds_hrdps_split):
    canrcm_times = []
    for t in ds_canrcm_split.time.values:
        canrcm_times.append(time_to_string(t))
    canrcm_times = np.array(canrcm_times)
    
    hrdps_times = []
    for t in ds_hrdps_split.time_counter.values:
        hrdps_times.append(time_to_string(t))
    hrdps_times = np.array(hrdps_times)
    
    common_times, canrcm_idx, hrdps_idx = np.intersect1d(canrcm_times,hrdps_times,return_indices=True)
    ds_canrcm_aligned = ds_canrcm_split.isel(time=canrcm_idx)
    ds_hrdps_aligned = ds_hrdps_split.isel(time_counter=hrdps_idx)
    
    return ds_canrcm_aligned, ds_hrdps_aligned

In [14]:
ds_canrcm_train, ds_hrdps_train = align_time_pair(ds_canrcm_train,ds_hrdps_train)
ds_canrcm_val, ds_hrdps_val = align_time_pair(ds_canrcm_val,ds_hrdps_val)
ds_canrcm_test, ds_hrdps_test = align_time_pair(ds_canrcm_test,ds_hrdps_test)
print("Training:")
print("CanRCM:", len(ds_canrcm_train.time))
print("HRDPS: ", len(ds_hrdps_train.time_counter))
print("Validation:")
print("CanRCM:", len(ds_canrcm_val.time))
print("HRDPS: ", len(ds_hrdps_val.time_counter))
print("Testing:")
print("CanRCM:", len(ds_canrcm_test.time))
print("HRDPS: ", len(ds_hrdps_test.time_counter))

Training:
CanRCM: 8760
HRDPS:  8760
Validation:
CanRCM: 2904
HRDPS:  2904
Testing:
CanRCM: 2912
HRDPS:  2912


In [15]:
X_canrcm_train = ds_canrcm_train["tas"].values.reshape(len(ds_canrcm_train.time),-1).astype(np.float32)
X_hrdps_train = ds_hrdps_train["tair"].values.reshape(len(ds_hrdps_train.time_counter),-1).astype(np.float32)
print("CanRCM training matrix shape:", X_canrcm_train.shape)
print("HRDPS training matrix shape:", X_hrdps_train.shape)
print("Missing CanRCM values:", np.isnan(X_canrcm_train).sum())
print("Missing HRDPS values:", np.isnan(X_hrdps_train).sum())

CanRCM training matrix shape: (8760, 1560)
HRDPS training matrix shape: (8760, 68096)
Missing CanRCM values: 0
Missing HRDPS values: 5175296


In [16]:
X_hrdps_train_raw = (ds_hrdps_train["tair"].squeeze(drop=True).values.reshape(len(ds_hrdps_train.time_counter), -1).astype(np.float32))
all_nan_times = np.isnan(X_hrdps_train_raw).all(axis=1)

In [17]:
keep_time = ~all_nan_times
ds_hrdps_train = ds_hrdps_train.isel(time_counter=keep_time)
ds_canrcm_train = ds_canrcm_train.isel(time=keep_time)

In [18]:
X_hrdps_train = (ds_hrdps_train["tair"].squeeze(drop=True).values.reshape(len(ds_hrdps_train.time_counter), -1).astype(np.float32))
X_canrcm_train = (ds_canrcm_train["tas"].squeeze(drop=True).values.reshape(len(ds_canrcm_train.time), -1).astype(np.float32))

In [19]:
valid_hrdps_cells = np.isfinite(X_hrdps_train).all(axis=0)
X_hrdps_train = X_hrdps_train[:, valid_hrdps_cells]

In [20]:
print("CanRCM matrix shape:", X_canrcm_train.shape)
print("HRDPS matrix shape: ", X_hrdps_train.shape)

CanRCM matrix shape: (8684, 1560)
HRDPS matrix shape:  (8684, 68096)


In [21]:
# Fit PCA
pca_canrcm = PCA()
pca_canrcm.fit(X_canrcm_train)
pca_hrdps = PCA()
pca_hrdps.fit(X_hrdps_train)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",None
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized S

In [22]:
canrcm_cumulative_variance = np.cumsum(pca_canrcm.explained_variance_ratio_)
hrdps_cumulative_variance = np.cumsum(pca_hrdps.explained_variance_ratio_)
thresholds = [0.90, 0.95, 0.98, 0.99]
print("CanRCM PCs")
for threshold in thresholds:
    n_pcs = np.argmax(canrcm_cumulative_variance >= threshold) + 1
    print(f"{threshold * 100:.0f}% variance explained by " f"{n_pcs} PCs")

print("HRDPS PCs")
for threshold in thresholds:
    n_pcs = np.argmax(hrdps_cumulative_variance >= threshold) + 1
    print(f"{threshold * 100:.0f}% variance explained by "f"{n_pcs} PCs")

CanRCM PCs
90% variance explained by 2 PCs
95% variance explained by 5 PCs
98% variance explained by 20 PCs
99% variance explained by 52 PCs
HRDPS PCs
90% variance explained by 2 PCs
95% variance explained by 6 PCs
98% variance explained by 28 PCs
99% variance explained by 113 PCs


In [23]:
n_hrdps_pcs = np.argmax(hrdps_cumulative_variance >= 0.98) + 1

In [24]:
X_canrcm_val = (
    ds_canrcm_val["tas"]
    .values
    .reshape(len(ds_canrcm_val.time), -1)
    .astype(np.float32)
)

X_hrdps_val = (
    ds_hrdps_val["tair"]
    .values
    .reshape(len(ds_hrdps_val.time_counter), -1)
    .astype(np.float32)
)

# Keep exactly the same HRDPS grid cells that were kept for training
X_hrdps_val = X_hrdps_val[:, valid_hrdps_cells]

# Remove validation timestamps containing missing values
valid_val_times = (
    np.isfinite(X_canrcm_val).all(axis=1)
    & np.isfinite(X_hrdps_val).all(axis=1)
)

X_canrcm_val = X_canrcm_val[valid_val_times]
X_hrdps_val = X_hrdps_val[valid_val_times]

print("CanRCM validation matrix shape:", X_canrcm_val.shape)
print("HRDPS validation matrix shape:", X_hrdps_val.shape)
print("Removed validation timestamps:", (~valid_val_times).sum())

CanRCM validation matrix shape: (2842, 1560)
HRDPS validation matrix shape: (2842, 68096)
Removed validation timestamps: 62


In [25]:
canrcm_train_scores_all = pca_canrcm.transform(X_canrcm_train)
canrcm_val_scores_all = pca_canrcm.transform(X_canrcm_val)

hrdps_train_scores = pca_hrdps.transform(X_hrdps_train)[:, :n_hrdps_pcs]
hrdps_val_scores = pca_hrdps.transform(X_hrdps_val)[:, :n_hrdps_pcs]

print("CanRCM training scores shape:", canrcm_train_scores_all.shape)
print("HRDPS training scores shape:", hrdps_train_scores.shape)

CanRCM training scores shape: (8684, 1560)
HRDPS training scores shape: (8684, 28)


In [41]:
max_canrcm_pcs = min(30, canrcm_train_scores_all.shape[1])

r2_matrix = np.zeros((n_hrdps_pcs, max_canrcm_pcs))

for k in range(1, max_canrcm_pcs + 1):
    
    X_train_k = canrcm_train_scores_all[:, :k]
    X_val_k = canrcm_val_scores_all[:, :k]
    
    model_k = LinearRegression()
    model_k.fit(X_train_k, hrdps_train_scores)
    
    hrdps_val_scores_pred_k = model_k.predict(X_val_k)
    
    r2_matrix[:, k - 1] = r2_score(
        hrdps_val_scores,
        hrdps_val_scores_pred_k,
        multioutput="raw_values"
    )

In [42]:
best_canrcm_pc_counts = np.argmax(r2_matrix, axis=1) + 1
best_val_r2_values = np.max(r2_matrix, axis=1)

pc_selection_results = pd.DataFrame({
    "HRDPS PC": np.arange(1, n_hrdps_pcs + 1),
    "Selected number of CanRCM PCs": best_canrcm_pc_counts,
    "Best validation R²": best_val_r2_values
})

print(pc_selection_results.to_string(index=False))

 HRDPS PC  Selected number of CanRCM PCs  Best validation R²
        1                             29            0.983185
        2                             29            0.856187
        3                             29            0.677575
        4                             30            0.780873
        5                             30            0.547586
        6                             17           -0.144067
        7                             21            0.186321
        8                             30            0.405677
        9                             30            0.422068
       10                             30            0.418112
       11                             30            0.354347
       12                             30            0.222839
       13                             30            0.274748
       14                             30            0.278333
       15                             30            0.154146
       16               

In [43]:
hrdps_val_scores_pred = np.zeros_like(hrdps_val_scores)

individual_models = []
for pc_index in range(n_hrdps_pcs):
    k = best_canrcm_pc_counts[pc_index]
    model_pc = LinearRegression()
    model_pc.fit(
        canrcm_train_scores_all[:, :k],
        hrdps_train_scores[:, pc_index]
    )
    hrdps_val_scores_pred[:, pc_index] = model_pc.predict(
        canrcm_val_scores_all[:, :k]
    )
    individual_models.append(model_pc)

In [44]:
hrdps_val_scores_pred

array([[-2.3380925e+03, -7.6436859e+01, -7.2744331e+01, ...,
         9.8605967e+00, -1.2847455e+01, -1.8010794e+01],
       [-2.6799771e+03, -3.3252567e+02, -1.8275866e+01, ...,
         1.9978077e+01,  3.7662048e+00, -7.9180536e+00],
       [-2.6106943e+03, -3.9652731e+02,  4.1425148e+01, ...,
         2.4731602e+01,  7.9611712e+00,  2.8979435e+00],
       ...,
       [-1.4837445e+03,  4.3346606e+02, -1.9730751e+01, ...,
         1.0305462e+00,  3.3507949e-01,  6.4216442e+00],
       [-1.4494919e+03,  3.8764264e+02,  3.6740032e+01, ...,
        -5.2339494e-01,  8.0102080e-01,  1.2084429e+00],
       [-1.4901388e+03,  3.3560770e+02,  5.8674610e+01, ...,
         4.6910350e-03, -4.2508707e+00,  7.2298865e+00]],
      shape=(2842, 28), dtype=float32)

In [46]:
# Create test matrices
X_canrcm_test = (
    ds_canrcm_test["tas"]
    .squeeze(drop=True)
    .values
    .reshape(len(ds_canrcm_test.time), -1)
    .astype(np.float32)
)

X_hrdps_test = (
    ds_hrdps_test["tair"]
    .squeeze(drop=True)
    .values
    .reshape(len(ds_hrdps_test.time_counter), -1)
    .astype(np.float32)
)

# Keep the same HRDPS grid cells used during training
X_hrdps_test = X_hrdps_test[:, valid_hrdps_cells]

# Remove test timestamps containing missing values
valid_test_times = (
    np.isfinite(X_canrcm_test).all(axis=1)
    & np.isfinite(X_hrdps_test).all(axis=1)
)

X_canrcm_test = X_canrcm_test[valid_test_times]
X_hrdps_test = X_hrdps_test[valid_test_times]

# Convert test data into PCA scores
canrcm_test_scores_all = pca_canrcm.transform(X_canrcm_test)

hrdps_test_scores = (
    pca_hrdps.transform(X_hrdps_test)[:, :n_hrdps_pcs]
)

# Combine training and validation scores
canrcm_trainval_scores_all = np.vstack([
    canrcm_train_scores_all,
    canrcm_val_scores_all
])

hrdps_trainval_scores = np.vstack([
    hrdps_train_scores,
    hrdps_val_scores
])

# Predict each HRDPS PC using its selected number of CanRCM PCs
hrdps_test_scores_pred = np.zeros_like(hrdps_test_scores)

for pc_index in range(n_hrdps_pcs):
    
    k = best_canrcm_pc_counts[pc_index]
    
    model_pc = LinearRegression()
    
    model_pc.fit(
        canrcm_trainval_scores_all[:, :k],
        hrdps_trainval_scores[:, pc_index]
    )
    
    hrdps_test_scores_pred[:, pc_index] = model_pc.predict(
        canrcm_test_scores_all[:, :k]
    )

# Calculate final test R²
test_r2_per_pc = r2_score(
    hrdps_test_scores,
    hrdps_test_scores_pred,
    multioutput="raw_values"
)

test_r2_average = r2_score(
    hrdps_test_scores,
    hrdps_test_scores_pred,
    multioutput="uniform_average"
)

test_r2_weighted = r2_score(
    hrdps_test_scores,
    hrdps_test_scores_pred,
    multioutput="variance_weighted"
)

print("Test R² for each HRDPS PC:")
print(test_r2_per_pc)

print("\nAverage test R²:")
print(test_r2_average)

print("\nVariance-weighted test R²:")
print(test_r2_weighted)

Test R² for each HRDPS PC:
[  0.9825337    0.6322423    0.73328286   0.6589162   -0.14146113
 -14.395474    -1.1774771   -2.2326279   -1.0480261   -0.98984873
  -0.3115623   -0.02580893  -0.24788356  -0.35446608  -0.304572
   0.03142893  -0.02725804  -0.57742286  -0.16566515  -0.30941248
  -0.43294644   0.04012197  -0.03264391  -0.12378073  -0.13399935
  -0.23725653  -0.69765925  -0.05115402]

Average test R²:
-0.7478528618812561

Variance-weighted test R²:
0.8787989616394043
